## Lab 04

## Zadanie 1 (Igor Sieradzki)

Funkcja <t>flow_rate</i> do modyfikacji

In [5]:
def flow_rate(weight, time, units_per_kg = 1, period = 1):
    """ Funkcja wylicza ile wagi produktu przybyło/ubyło w jednostce czasu """
    return weight / time * units_per_kg / period

weight = 0.5
time = 3

In [6]:
flow = flow_rate(weight, time)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [4]:
"""Zmodyfikuj funckje <i>flow_rate</i> tak, aby poniższe wywołania działały poprawnie."""

'Zmodyfikuj funckje <i>flow_rate</i> tak, aby poniższe wywołania działały poprawnie.'

In [7]:
flow = flow_rate(weight, time, period=60, units_per_kg=1000)
print("{} grams per minute".format(flow))

2.7777777777777777 grams per minute


In [8]:
flow = flow_rate(weight, time, period=1, units_per_kg=1)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [9]:
flow = flow_rate(weight, time)
print("{0:.3} grams per minute".format(flow))

0.167 grams per minute


Wyjaśnić czemu poniższe rzucanie błędu jest porządane

In [39]:
try:
    flow = flow_rate(weight, time, 3600, 2.2)
except TypeError:
    print(True)

## Zadanie 2 (Igor Sieradzki)

Dopisz definicje dekoratora _timeit_, który wypisze na wyjscie standardowe czas wywołania udekorowanej funckji np.

```
@timeit
def foo(x):
    return x**2
    
r = foo(2)
```
wypize: <br>
    `Function foo took: 0.00001 seconds` <br>
oraz zwróci 4

In [44]:
from time import time

def timeit(func):
    """ Wypisuje czas wywołania udekorowanej funckji """
    def wrapper(argument):
        base_time = time()
        func(argument)
        finish_time = time()
    
        print(f"Function {func.__name__} took: {finish_time - base_time} seconds")
        return 4
    
    return wrapper

@timeit
def squares_list(n):
    squares = []
    for i in range(n):
        squares.append(i ** 2)
    return squares

@timeit
def squares_comprehension(n):
    return [i ** 2 for i in range(n)]

@timeit
def squares_map(n):
    return map(lambda x: x**2, range(n))

n = 1000000
l = squares_list(n)
c = squares_comprehension(n)
m = squares_map(n)

Function squares_list took: 0.17366981506347656 seconds
Function squares_comprehension took: 0.15913724899291992 seconds
Function squares_map took: 3.0994415283203125e-06 seconds


## Zadanie 3 (Igor Sieradzki)

Dopisz definicje dekoratora _derivate_ wg. instrukcji w _docstringu_

In [58]:
import sys
sys.float_info.epsilon # epsilon maszynowy


def derivate(epsilon=None):
    """
    Zwraca pochodną funkcji w punkcie, wg. wzoru f'(x) = [f(x+h) - f(x)]/h, 
    gdzie h jest parametrem dekoratora, jeśli nie zostanie podany, należy przyjąć 1000 * epsilon maszynowy
    """
    if epsilon is None:
        epsilon = 1000 * sys.float_info.epsilon

    def wrapper(function):
        def wrapper_wrapper(x):
            return (function(x + epsilon) - function(x)) / epsilon

        return wrapper_wrapper
    return wrapper

@derivate(0.01)
def f(x):
    return x*x

@derivate(0.00001)
def g(x):
    return x*x*x+3

def test(a, b, eps=1):
    return abs(round(a)-round(b)) < eps

print(test(f(100), 200.0))
print(round(f(0)) == 0.0)

print(test(g(100), 30000.0))
print(round(g(0)) == 0.0)

from random import random
for x in [random()*1000. for _ in range(20)]:
    print(f(x), 2*x, '\t', test(f(x), 2*x))
    print(g(x), 3*x**2, '\t', test(g(x), 3*x**2))

True
True
True
True
582.6455462054582 582.6355462058814 	 True
254598.14295172688 254598.1347769693 	 True
847.3682116775308 847.3582116785236 	 True
538511.9646787643 538511.9541742692 	 True
203.67000517508131 203.66000517488493 	 True
31108.051328919824 31108.048280875617 	 True
1486.0959067940712 1486.0859067959866 	 True
1656338.5128974912 1656338.491783237 	 False
1059.632881282596 1059.6228812816814 	 True
842100.504040718 842100.4879017691 	 False
1561.4845881820656 1561.4745881815413 	 True
1828652.1852016447 1828652.1671525356 	 True
913.7157533725258 913.7057533756405 	 True
626143.6656117439 626143.6528138102 	 True
1676.425911881961 1676.4159118831394 	 True
2107777.762413025 2107777.732211233 	 True
834.068001669948 834.0580016704884 	 True
521739.5737767219 521739.56261292624 	 True
524.7875443776138 524.7775443781085 	 True
206543.61099004743 206543.60331263818 	 True
33.30289570563423 33.29289570562843 	 True
831.3131777867965 831.312678349389 	 True
1714.0107401879504

## Zadanie 4 (Igor Sieradzki)

Dopisz definicje dekoratora _accepts_ wg. instrukcji w _docstringu_

In [62]:
def accepts(*types):
    """Sprawdza czy udekorowanej funckji zostały podane odpowiednie parametry zdefiniowane 
       w argumentach dekoratora"""

    def decorator(function):
        code = function.__code__
        names = code.co_varnames[:code.co_argcount]          # ('base', 'exp')
        expected = dict(zip(names, types))

        def wrapper(*args, **kwargs):
            given = dict(zip(names, args))                   # positional args -> their names
            given.update(kwargs)                             # keyword args are already named
            for name, value in given.items():
                if name in expected and not isinstance(value, expected[name]):
                    raise TypeError(f"argument '{name}' of {function.__name__}() must be "
                                    f"{expected[name].__name__}, not {type(value).__name__}")
            return function(*args, **kwargs)
        return wrapper
    return decorator


@accepts(str)
def capitalize(word):
    return word[0].upper() + word[1:]

print(capitalize('ola') == 'Ola')

try:
    capitalize(2)
except TypeError:
    print(True)

@accepts(float, int)
def static_pow(base, exp):
    return base ** exp 

print(static_pow(2., 2) == 4.)
print(static_pow(2., exp=2) == 4.)
print(static_pow(base=2., exp=2) == 4.)

try:
    static_pow('x', 10)
except TypeError:
    print(True)
    
try:
    static_pow(2, 2.2)
except TypeError:
    print(True)

True
True
True
True
True
True
True


## Zadanie 5 (Igor Sieradzki)

Dopisz definicje dekoratora _returns_ wg. instrukcji w _docstringu_

In [66]:
def returns(*types):
    """Sprawdza czy udekorowana funkcja zwraca poprawne argumenty, zdefiniowane w parametrach dekoratora"""

    def wrapper(function):
        def wrapper_wrapper(*arguments, **dict_arguments):
            result = function(*arguments, **dict_arguments)

            for desired_type, returned_value in zip(types, result):
                if type(returned_value) != desired_type:
                    raise TypeError

            return result

        return wrapper_wrapper

    return wrapper


@returns(str)
def str_only_identity(word):
    return word

print(str_only_identity('hello') == 'hello')

try:
    str_only_identity(10)
except TypeError:
    print(True)
    
@returns(int, int)
def split_indices(x):
    return x[0], x[1]

print(split_indices(x=[6,9]) == (6,9))

try:
    split_indices('AB')
except TypeError:
    print(True)

True
True
True
True


## Zadanie 6 (Igor Sieradzki)
Stwórz dekorator cached służący do cachowania wywołań dowolnej funkcji, tzn. chcemy by:
* wywołanie funkcji z określonymi argumentami miało miejsce tylko raz
* funkcja mogła przyjmować dowolną liczbę nazwanych i nienazwanych argumentów
* nie musi reagować poprawnie na domyślne argumenty, tzn. wywołanie funkcji z domyślnymi argumentami a podanie dokładnie takich samych może być traktowane jako dwa różne wywołania
* na opakowanej funkcji można wywołać `.cache_reset()`, który usunie cache z pamięci
* wywołanie `.cache_status()` zwraca string z opisem w postaci: <br>
    `Function FUNCTION_NAME called X times, evaluated Y times`

In [67]:
from random import random

def cached(function):
    cache = {}
    calls = 0
    evaluations = 0

    def wrapper(*args, **kwargs):
        nonlocal calls, evaluations
        calls += 1
        key = (args, frozenset(kwargs.items()))
        if key not in cache:
            evaluations += 1
            cache[key] = function(*args, **kwargs)
        return cache[key]

    def cache_reset():
        cache.clear()

    def cache_status():
        return f"Function {function.__name__} called {calls} times, evaluated {evaluations} times"

    wrapper.cache_reset = cache_reset
    wrapper.cache_status = cache_status
    return wrapper

@cached
def foo(x, y=1, z=4):
    return random()
        
print(foo(3) == foo(3))
print(foo(4) == foo(4))
print(foo(3, z=-1, y=3) == foo(3, y=3, z=-1))
print(foo(3) != foo(x=3))
a = foo(3)
foo.cache_reset()
print(a != foo(3))
print(foo.cache_status() == 'Function foo called 10 times, evaluated 5 times')

True
True
True
True
True
True


## Zadanie 7 (Krzysztof Hajto)

Napisz dekorator który będzie robić n-krotne złożenie funkcji, gdzie n jest parametrem dekoratora

In [68]:
def zlozenie(n):

    def wrapper(function):
        def wrapper_wrapper(x):
            result = function(x)
            for i in range(n - 1):
                result = function(result)

            return result

        return wrapper_wrapper
    return wrapper

@zlozenie(3)
def f1(x):
    return x+1

@zlozenie(2)
def f2(x):
    return x*x

@zlozenie(5)
def f3(word):
    return "".join(chr(ord(l)+1) for l in word)

print(f1(2)==5)
print(f2(3)==81)
print(f3("alamakota")=="fqfrfptyf")

True
True
True


## Zadanie 8 (Krzysztof Hajto)

Python nie ma wbudowanej instrukcji switch. Ale posiada anonimowe funkcje oraz słowniki. Zaimplementuj poniższy switch w postaci słownika funkcji.
`
int my_function(x, y) {


    switch(x) {
        case 1: return y*y;
        case 2: return x+y;
        case 3: return x*y;
        case 4: return 0;
    }
}
`

PS. Nigdy nie róbcie tego w faktycznym kodzie :)

In [ ]:
def my_function(x, y):
    if x == 1:
        return y * y
    elif x == 2:
        return x + y
    elif x == 3:
        return x * y
    elif

print(my_function(1,3)==9)
print(my_function(2,4)==6)
print(my_function(3,1)==3)
print(my_function(4,9)==0)